# Práctica 1: Modelado y Simulación Topológica

## Cálculo de Homología Simplicial
Vamos a estudiar como calcular homología simplicial sobre un cuerpo mediante algunas librerías de Python. 

Empezamos instalando los módulos necesarios. 
Sobretodo, vamos a tener que usar la librería `Numpy` (https://numpy.org/) de python que sirve para trabajar con matrices.
Por otro lado, para los cálculos con complejos simpliciales y homología necesitaremos utilizar la librería `GUDHI` (https://gudhi.inria.fr/) 

In [ ]:
pip install numpy gudhi

También utilizaremos las liberías `NetworkX` y `Matplotlib` para visualizar algunos complejos simpliciales.

In [ ]:
pip install networkx matplotlib

Por otro lado, vamos a instalar el módulo PHAT (https://www.sciencedirect.com/science/article/pii/S0747717116300098) que nos permitirá trabajar con las matrices sobre $\mathbb{Z}_2$. Para esto utilizaremos una versión algo modificada.

In [ ]:
!pip install setuptools pybind11
!pip install --no-build-isolation git+https://bitbucket.org/atorras1618/phat.git

### Ejemplo 1: 
Vamos a crear un complejo simplicial mediante la estructura de datos `simplex_tree` de GUDHI. Para esto, importaremos el objeto simplex tree con el alias `st` y le añadiremos los símplices del siguiente complejos simplicial:

![Ejemplo 1](ejemplo-1.png)

In [ ]:
import gudhi
st = gudhi.SimplexTree()
# Añadimos los símplices de dimensión 0
st.insert([0])
st.insert([1])
st.insert([2])
st.insert([3])
# Añadimos los símplices de dimensión 1
st.insert([0,1])
st.insert([0,2])
st.insert([1,2])
st.insert([1,3])
st.insert([2,3])
# Añadimos el símplice de dimensión 2
st.insert([1,2,3])

Cada vez que insertamos un símplice en el objeto `st`, el método devuelve `True` cuando la inserción se ha realizado con éxito. Por otro lado, podemos comprobar leer todos los símplices que hemos insertado mediante el método `get_simplices()`

In [ ]:
print(list(st.get_simplices()))

In [ ]:
def simplices_dictionary(st):
    dict_spx = {}
    for i in range(st.dimension()+1):
        dict_spx[i] = []
    for simplex, filt in st.get_simplices():
        simplex_dim = len(simplex)-1
        dict_spx[simplex_dim].append(simplex)
    return dict_spx

In [ ]:
dict_spx = simplices_dictionary(st)
print(dict_spx)

Nótese que Gudhi guarda los símplices como pares (símplice, valor de filtración). El valor por defecto es `0.0`.

In [ ]:
print(f"El complejo simplicial tiene {st.num_vertices()} vértices y un total de {st.num_simplices()} símplices.")

Vamos ahora a visualizar el complejo simplicial, tal como lo hemos visto al principio. Para esto crearemos una función que representa un complejo simplicial en el plano 

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.collections import PolyCollection

def plot_2D_simplicial_complex(st, pos=None, figsize=(6,6), facecolors='skyblue', alpha=0.5):
    G = nx.Graph(simplices_dictionary(st)[1])
    if pos is None:
        pos = nx.spring_layout(G)
    fig, ax = plt.subplots(figsize=figsize)
    triangles = []
    options = {
        "font_size": 16,
        "node_size": 1000,
        "node_color": "white",
        "edgecolors": "black",
        "linewidths": 5,
        "width": 5,
    }
    for simplex, filt in st.get_simplices():
        if len(simplex) == 3:
            triangles.append(simplex)

    # Plot Triangles (Faces)
    if triangles:
        coords = np.array([pos[i] for i in pos.keys()])
        poly_coords = [coords[nodes] for nodes in triangles]
        face_col = PolyCollection(poly_coords, edgecolors='none', facecolors=facecolors, alpha=alpha)
        ax.add_collection(face_col) 
        
    nx.draw(G, with_labels=True, pos=pos, **options)

In [ ]:
plot_2D_simplicial_complex(st)

In [ ]:
pos = {0: [0, 0], 1: [1, 0], 2: [0, 1], 3: [1, 1]}
plot_2D_simplicial_complex(st, pos=pos, figsize=(4,4))
plt.tight_layout()
plt.savefig("ejemplo-1.png")

In [ ]:
st.betti_numbers()

In [ ]:
list(st.get_boundaries([1,2,3]))